In [0]:
import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, DateType
from pyspark.sql.window import Window

In [0]:
# Storage configuration
STORAGE_ACCOUNT = "stockpipelinedl"
CONTAINER_BRONZE = "bronze"
CONTAINER_SILVER = "silver"
CONTAINER_GOLD = "gold"

# Secure credential retrieval
def get_storage_key():
    return dbutils.secrets.get(scope="stock-pipeline", key="azure-storage-key")

# Connect Spark to Azure Storage
def configure_spark_azure():
    storage_key = get_storage_key()
    spark.conf.set(
        f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        storage_key
    )
    print(f"Spark is connected to Azure Storage ({STORAGE_ACCOUNT})")
    return storage_key

# ADLS path builder
def get_adls_path(container, subfolder="stock_data"):
    return f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{subfolder}/"

# Dynamic date
def get_processing_date(fallback_days=4):
    try:
        date = dbutils.widgets.get("execution_date")
        print(f"Date from Airflow: {date}")
        return date
    except:
        date = (datetime.datetime.utcnow() - datetime.timedelta(days=fallback_days)).strftime('%Y-%m-%d')
        print(f"Date from fallback ({fallback_days} days ago): {date}")
        return date

# Last trading day calculator
def get_last_trading_day():
    """Automatically skips weekends"""
    today = datetime.datetime.utcnow()
    yesterday = today - datetime.timedelta(days=1)
    if yesterday.weekday() == 5:    # Saturday
        yesterday -= datetime.timedelta(days=1)
    elif yesterday.weekday() == 6:  # Sunday
        yesterday -= datetime.timedelta(days=2)
    return yesterday.strftime('%Y-%m-%d')

# DQ check runner
def run_dq_checks(df, checks):
    """
    Runs a list of DQ checks and prints a report.
    Returns True if all pass, False otherwise.
    """
    results = []
    for check_name, failed_count in checks:
        results.append({
            "check": check_name,
            "passed": failed_count == 0,
            "failed_count": failed_count
        })

    print("=" * 50)
    print("DATA QUALITY REPORT")
    print("=" * 50)
    all_passed = True
    for r in results:
        status = "PASS!" if r["passed"] else "FAIL!"
        print(f"{status} | {r['check']} | Failed: {r['failed_count']}")
        if not r["passed"]:
            all_passed = False
    print("=" * 50)
    print(f"Overall: {'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED'}")
    return all_passed

print("Utils loaded successfully")